In [1]:
# Environment Setup
!git clone https://github.com/anikanb-32/musicandmemory.git
%cd musicandmemory
!cd /content/musicandmemory && git pull
!pip install -r requirements.txt

Cloning into 'musicandmemory'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 113 (delta 60), reused 44 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 45.01 KiB | 940.00 KiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/musicandmemory
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 k

In [3]:
# Connect to Google Drive and OpenAI API
from google.colab import userdata, drive
import os, json, time
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/GenAI Final Project Music&Memory/'

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
client = OpenAI()

Mounted at /content/drive


In [4]:
# Test to see if imported files are empty
!cat /content/musicandmemory/src/retrieval.py
!cat /content/musicandmemory/src/generation.py
!cat /content/musicandmemory/configs/prompts.py

import faiss
import numpy as np
import pandas as pd
from openai import OpenAI

client = OpenAI()

def load_retrieval_system(index_path, kb_path):
    """Load the FAISS index and knowledge base."""
    index = faiss.read_index(index_path)
    df = pd.read_csv(kb_path)
    return index, df

def retrieve(query, index, df, k=20):
    """Retrieve top-k songs for a query string."""
    # Embed the query
    response = client.embeddings.create(
        input=[query], model="text-embedding-3-small"
    )
    query_vec = np.array([response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(query_vec)

    # Search
    scores, indices = index.search(query_vec, k)

    # Return results
    results = df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    return results
from openai import OpenAI
import json

client = OpenAI()

def generate_playlist(profile, retrieved_songs_df, prompt_template):
    """Generate a playlist and caregiver cards from profile + retrieved song

In [5]:
# Test to see if imported files are working
from src.profiling import generate_queries, profile_to_context
from src.retrieval import load_retrieval_system, retrieve
from src.generation import generate_playlist
from configs.prompts import GENERATION_PROMPT


print('All imports succeed!')

All imports succeed!


In [6]:
# Load the retrieval system
index, df = load_retrieval_system(DATA_PATH + 'songs.index', DATA_PATH + 'knowledge_base.csv')

# Define a patient
patient = {
    "name": "James Wilson",
    "birth_year": 1948,
    "hometown": "Detroit, Michigan",
    "cultural_background": "African American",
    "life_events": [
        {"year": 1966, "event": "Graduated from Cass Tech High School"},
        {"year": 1968, "event": "Drafted into the Vietnam War"},
        {"year": 1971, "event": "Married Dorothy in Detroit"},
        {"year": 1975, "event": "First child born"},
    ]
}

# Retrieved songs based on profile
retrieved, queries = profile_to_context(patient, index, df)
print(f"Retrieved {len(retrieved)} unique songs")

# Generate playlist
result = generate_playlist(patient, retrieved, GENERATION_PROMPT)

# Display
print("\n=== PLAYLIST ===")
for song in result["playlist"]:
    print(f"  {song['rank']}. {song['song']} — {song['artist']} ({song['year']})")
    print(f"     Reason: {song['relevance']}\n")

print("\n=== CAREGIVER CARDS ===")
for card in result["caregiver_cards"]:
    print(f"  Song: {card['song']}")
    print(f"  Prompt: {card['prompt']}\n")


Generated 8 queries:
  - popular Motown hits in Detroit 1963-1973
  - top R&B songs in Detroit 1966
  - Vietnam War era songs popular among African Americans 1968
  - wedding songs popular in Detroit 1971
  - soul and funk hits in Detroit 1975
  - songs played on Detroit radio stations 1963-1973
  - African American artists popular in Detroit 1963-1973
  - graduation year hits 1966 Detroit
Retrieved 20 unique songs

=== PLAYLIST ===
  1. Respect — Aretha Franklin (1967)
     Reason: This song is an iconic anthem from an influential African American artist from Detroit, resonating with James's cultural background and hometown.

  2. Dancing In The Street — Martha & The Vandellas (1964)
     Reason: A classic Motown hit from Detroit, it captures the vibrant music scene of James's youth and his hometown.

  3. Soul Man — Sam & Dave (1967)
     Reason: This soul classic aligns with James's cultural background and was popular during his late teens.

  4. War — Edwin Starr (1970)
     Reason